# 이미지 생성 파이프라인 v0 (관통판)

- **목적**: 사진 → 누끼 → 배경 → 합성 → 문구까지 전체 흐름을 먼저 연결 (품질은 이후 단계)
- **임시 부품**: 배경(그라데이션 → 추후 SDXL 교체), 문구(고정 문장 → 추후 GPT 연동)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from rembg import remove, new_session

def show(im, size=380):
    """표시용 축소 미리보기"""
    p = im.copy()
    p.thumbnail((size, size))
    return p

In [ ]:
img = Image.open("테스트사진.jpg")
sess = new_session("isnet-general-use")
cut = remove(img, session=sess, post_process_mask=True)
show(cut)

In [ ]:
# 임시 배경: 크림→주황 그라데이션 (나중에 SDXL이 이 자리를 대체)
W = H = 1080
top, bottom = (255, 244, 228), (240, 160, 90)

bg = Image.new("RGB", (W, H))
d = ImageDraw.Draw(bg)
for y in range(H):
    t = y / H
    d.line([(0, y), (W, y)],
           fill=tuple(int(top[i] + (bottom[i] - top[i]) * t) for i in range(3)))

prod = cut.copy()
prod.thumbnail((880, 880))
canvas = bg.convert("RGBA")
canvas.alpha_composite(prod, ((W - prod.width) // 2, H - prod.height - 100))
show(canvas)

In [ ]:
d = ImageDraw.Draw(canvas)
big = ImageFont.truetype("C:/Windows/Fonts/malgunbd.ttf", 76)
mid = ImageFont.truetype("C:/Windows/Fonts/malgunbd.ttf", 52)

d.text((W // 2, 140), "오늘의 특선 오므라이스", font=big, fill=(70, 45, 20), anchor="mm")
d.text((W // 2, 235), "9,900원", font=mid, fill=(210, 85, 20), anchor="mm")

final = canvas.convert("RGB")
show(final)

In [ ]:
from PIL import Image

from app_core.background import remove_background
from app_core.compose import compose_ad

img = Image.open("테스트사진.jpg")   # notebooks 폴더의 그 사진
cut = remove_background(img)

# 아침에 앱에서 받은 진짜 문구 카드로 테스트
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!")
ad.save("미리보기_두줄.png")

small = ad.copy()
small.thumbnail((360, 360))
small

In [ ]:
compose_ad(cut, "맛있는 크로플을 아주 여유롭게 경험하세요", "독서실처럼 조용한 공간, 코드잇스터디카페에서 만나요")

small_compose = ad.copy()
small_compose.thumbnail((360, 360))
small_compose